In [ ]:
#| default_exp phone

# On a phone

> Kokoro's decoder on a Pixel 10a through LiteRT: what the converter gets wrong, and the CPU against the GPU.

Notebook 103 got Kokoro's decoder onto the Mac's GPU and Neural Engine. This one puts it on an Android phone, a Pixel
10a, through LiteRT (Google's runtime, formerly TensorFlow Lite), which reaches the CPU through XNNPACK and the GPU
through its GPU delegate.

It needs the phone plugged in with USB debugging on, so it's marked `skip_exec`. It also needs `uv`, because Google's
PyTorch-to-LiteRT converter pins an older `typing-extensions` than Jupyter will run with, so it runs in a throwaway
environment of its own.

In [ ]:
#| export
import re
import shutil
import subprocess
import types
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from kokoro.istftnet import AdaIN1d, UpSample1d

from text_to_speech.core import DATA
from text_to_speech.deploy import Body, HarmonicSource, decoder_inputs, exportable_decoder
from text_to_speech.pretrained import say

/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 21:39:39.557000 93085 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 21:39:39.600000 93085 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1101 21:39:39.654000 93085 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclas

In [ ]:
import pandas as pd

## The phone

`adb` talks to it over USB.

In [ ]:
#| export
ADB = shutil.which("adb") or str(Path.home() / "Library/Android/sdk/platform-tools/adb")
PHONE = DATA / "phone"


def adb(*args):
    "Run an adb command. Returns what it printed, stdout and stderr together."
    r = subprocess.run([ADB, *args], capture_output=True, text=True, check=False)  # adb's exit code says little
    return r.stdout + r.stderr

In [ ]:
for prop in ["ro.product.model", "ro.soc.model", "ro.build.version.release", "ro.product.cpu.abi"]:
    print(f"{prop:<26} {adb('shell', 'getprop', prop).strip()}")

ro.product.model           Pixel 10a
ro.soc.model               Tensor G4
ro.build.version.release   17
ro.product.cpu.abi         arm64-v8a


## Into LiteRT

`to_tflite` saves a module with `torch.export`, converts it with `litert-torch` in its own environment (`uv run
--no-project`, which uv builds once and caches), and runs the result on the Mac's CPU to check it against PyTorch.
LiteRT models have fixed shapes, so like the Core ML ones in notebook 103 they're made for one number of frames.

In [ ]:
#| export
CONVERT = """
import sys, numpy as np, torch, litert_torch
pt2, io, out = sys.argv[1:]
io = np.load(io)
args = tuple(torch.tensor(io[f"arr_{i}"]) for i in range(len(io.files) - 1))
model = litert_torch.convert(torch.export.load(pt2).module(), args)
model.export(out)
print(np.abs(np.asarray(model(*args)).reshape(io["out"].shape) - io["out"]).max())
"""


@torch.no_grad()
def to_tflite(module, args, name):
    "Convert `module`, at the shapes of `args`, to PHONE/name.tflite. Returns its largest error against PyTorch."
    PHONE.mkdir(parents=True, exist_ok=True)
    module = module.eval()
    torch.export.save(torch.export.export(module, args), PHONE / f"{name}.pt2")
    np.savez(PHONE / f"{name}.npz", *[a.numpy() for a in args], out=module(*args).numpy())
    uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
    cmd = [uv, "run", "-q", "--no-project", "--python", "3.13", "--with", "torch==2.14.0", "--with", "litert-torch"]
    r = subprocess.run(
        [*cmd, "python", "-c", CONVERT, PHONE / f"{name}.pt2", PHONE / f"{name}.npz", PHONE / f"{name}.tflite"],
        capture_output=True,
        text=True,
        check=True,
    )
    return float(r.stdout.strip().splitlines()[-1])

The decoder body from notebook 103, for 128 frames (3.2 s), with the inputs of a real sentence:

In [ ]:
T = 128
asr, F0, N, s = decoder_inputs(
    say("Speed comes from the voice itself, not from stretching the audio afterwards.").phonemes
)
dec = exportable_decoder()
with torch.no_grad():
    args = tuple(
        a.contiguous() for a in (asr[..., :T], F0[:, : 2 * T], N[:, : 2 * T], s, HarmonicSource(dec)(F0[:, : 2 * T]))
    )
    reference = Body(dec)(*args)
print(
    f"LiteRT vs PyTorch: largest error {to_tflite(Body(dec), args, 'first_try'):.3f}, on audio of RMS {reference.pow(2).mean().sqrt():.3f}"
)

LiteRT vs PyTorch: largest error 0.488, on audio of RMS 0.048


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/export/pt2_archive/_package.py:415: UserWarning: No complete tensor found in the group! Returning the first one. This may cause issues when your weights are not on CPU.
  complete_tensor = get_complete_tensor(group, weights_dict)


It converts, and it's wrong: an error ten times the size of the signal. Converting the body cut off after each stage
in turn (not shown) found the first stage that went wrong, and inside it the culprit: a transposed convolution with
padding. On its own:

In [ ]:
for name, layer in [
    ("kernel 3, stride 2, padding 1", torch.nn.ConvTranspose1d(64, 64, 3, stride=2, padding=1)),
    ("kernel 20, stride 10, padding 5", torch.nn.ConvTranspose1d(64, 32, 20, stride=10, padding=5)),
]:
    print(f"ConvTranspose1d, {name}: largest error {to_tflite(layer, (torch.randn(1, 64, 100),), 'layer'):.1e}")

ConvTranspose1d, kernel 3, stride 2, padding 1: largest error 4.1e+00
ConvTranspose1d, kernel 20, stride 10, padding 5: largest error 3.0e-07


One shape of transposed convolution converts wrongly, and the generator's happens not to. So the fix is to avoid
padded transposed convolutions altogether. That, and four more changes, gets the model through the converter and onto
the GPU. Each is the same maths written another way:

- **Padded transposed convolutions** become unpadded ones, then a crop of what the padding would have removed.
- **The depthwise transposed convolution** (1,090 channels, one per group) becomes zeros between samples, then an
  ordinary depthwise convolution with the kernel flipped. Left alone, the converter splits it into 1,090 separate
  operations.
- **Nearest-neighbour upsampling** becomes stacking and reshaping, and a **one-sample reflection pad** becomes
  copying that sample. Both otherwise become `GATHER_ND`, which the GPU doesn't support.
- **Instance norm** is written out as means and a square root.
- **The inverse STFT's window envelope**, built from a tensor of ones, gets folded into a constant the GPU delegate
  refuses as an input. For a fixed number of frames it's a constant anyway, so we precompute its reciprocal.

In [ ]:
#| export
class CroppedConvTranspose1d(torch.nn.Module):
    "A padded ConvTranspose1d as an unpadded one and a crop: the same maths, which LiteRT converts correctly."

    def __init__(self, ct):
        super().__init__()
        self.ct, self.crop = ct, (ct.padding[0], ct.padding[0] - ct.output_padding[0])
        ct.padding, ct.output_padding = (0,), (0,)

    def forward(self, x):
        y = self.ct(x)
        return y[..., self.crop[0] : y.shape[-1] - self.crop[1]]


class ZeroStuffedConv1d(torch.nn.Module):
    "A depthwise ConvTranspose1d as zeros between samples, then a depthwise Conv1d with the kernel flipped."

    def __init__(self, ct):
        super().__init__()
        C, k, (s,), (p,), (op,) = ct.in_channels, ct.kernel_size[0], ct.stride, ct.padding, ct.output_padding
        self.conv = torch.nn.Conv1d(C, C, k, groups=C, bias=ct.bias is not None)
        self.conv.weight.data = ct.weight.data.flip(-1)
        if ct.bias is not None:
            self.conv.bias.data = ct.bias.data
        self.stride, self.pad = s, (k - 1 - p, k - 1 - p + op)

    def forward(self, x):
        B, C, L = x.shape
        z = torch.stack([x, *[torch.zeros_like(x)] * (self.stride - 1)], -1).reshape(B, C, L * self.stride)
        return self.conv(F.pad(z[..., : (L - 1) * self.stride + 1], self.pad))


class Nearest2(torch.nn.Module):
    "F.interpolate(x, scale_factor=2, mode='nearest'), as stacking and reshaping."

    def forward(self, x):
        return torch.stack([x, x], -1).reshape(x.shape[0], x.shape[1], 2 * x.shape[2])


class ReflectOne(torch.nn.Module):
    "ReflectionPad1d((1, 0)): the second sample, copied in front."

    def forward(self, x):
        return torch.cat([x[..., 1:2], x], -1)


def _adain(self, x, s):
    "AdaIN1d with the instance norm written out as means and a square root."
    gamma, beta = torch.chunk(self.fc(s)[:, :, None], chunks=2, dim=1)
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    x = (x - mean) / torch.sqrt(var + self.norm.eps) * self.norm.weight[:, None] + self.norm.bias[:, None]
    return (1 + gamma) * x + beta


def _rewrite(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ConvTranspose1d) and child.groups > 1:
            setattr(module, name, ZeroStuffedConv1d(child))
        elif isinstance(child, torch.nn.ConvTranspose1d) and child.padding[0]:
            setattr(module, name, CroppedConvTranspose1d(child))
        elif isinstance(child, UpSample1d) and child.layer_type != "none":
            setattr(module, name, Nearest2())
        elif isinstance(child, torch.nn.ReflectionPad1d):
            setattr(module, name, ReflectOne())
        else:
            if isinstance(child, AdaIN1d):
                child.forward = types.MethodType(_adain, child)
            _rewrite(child)


def _fixed_envelope(stft, frames):
    "For a fixed number of frames the inverse STFT's window envelope is a constant: precompute its reciprocal."
    pad = stft.n_fft // 2
    envelope = F.conv_transpose1d(torch.ones(1, 1, frames), stft.window_squared, stride=stft.hop_length)
    stft.register_buffer("inv_envelope", 1 / envelope[..., pad:-pad])

    def inverse(self, magnitude, phase):
        real = F.conv_transpose1d(magnitude * torch.cos(phase), self.weight_backward_real, stride=self.hop_length)
        imag = F.conv_transpose1d(magnitude * torch.sin(phase), self.weight_backward_imag, stride=self.hop_length)
        return (real - imag)[..., pad:-pad] * self.inv_envelope

    stft.inverse = types.MethodType(inverse, stft)


def phone_decoder(T):
    "The decoder for T frames, rewritten for LiteRT and its GPU delegate: (HarmonicSource, Body)."
    dec = exportable_decoder()
    source = HarmonicSource(dec).eval()
    _rewrite(dec)
    _fixed_envelope(dec.generator.stft, T * 120 + 1)  # 120 source columns per frame
    return source, Body(dec).eval()

In [ ]:
source, body = phone_decoder(T)
with torch.no_grad():
    print(f"rewritten vs original in PyTorch: largest error {(body(*args) - reference).abs().max():.1e}")
print(f"LiteRT vs PyTorch: largest error {to_tflite(body, args, f'body{T}'):.1e}")
print(f"{(PHONE / f'body{T}.tflite').stat().st_size / 1e6:.0f} MB")

rewritten vs original in PyTorch: largest error 4.1e-06
LiteRT vs PyTorch: largest error 7.5e-06
214 MB


## Speed

Google publishes `benchmark_model`, a command-line tool that loads a LiteRT model on the phone and times it. We copy it
to `/data/local/tmp`, the phone's folder for command-line tools, along with the model.

In [ ]:
#| export
TOOL = "https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/android_aarch64_benchmark_model"
ON_PHONE = "/data/local/tmp"


def install(path):
    "Copy a file to the phone's tools folder, downloading the benchmark tool first if needed."
    path = Path(path)
    if path.name == "benchmark_model" and not path.exists():
        urllib.request.urlretrieve(TOOL, path)
    adb("push", str(path), f"{ON_PHONE}/{path.name}")
    adb("shell", "chmod", "+x", f"{ON_PHONE}/{path.name}")


def benchmark(model, *flags, runs=10):
    "Time `model` on the phone. Returns (seconds to load, seconds per inference) and the tool's full output."
    out = adb(
        "shell",
        f"{ON_PHONE}/benchmark_model",
        f"--graph={ON_PHONE}/{model}",
        f"--num_runs={runs}",
        "--warmup_runs=2",
        *flags,
    )
    m = re.search(r"Init: ([\d.e+]+).*Inference \(avg\): ([\d.e+]+)", out)
    if not m:
        raise RuntimeError(out[-2000:])
    return float(m[1]) / 1e6, float(m[2]) / 1e6, out

In [ ]:
install(PHONE / "benchmark_model")
install(PHONE / f"body{T}.tflite")
seconds, rows = T * 0.025, []
for name, flags in [
    ("CPU, 1 thread", ["--num_threads=1"]),
    ("CPU, 4 threads", ["--num_threads=4"]),
    ("CPU, 8 threads", ["--num_threads=8"]),
    ("GPU, 16-bit", ["--use_gpu=true"]),
    ("GPU, 32-bit", ["--use_gpu=true", "--gpu_precision_loss_allowed=false"]),
]:
    load, infer, _ = benchmark(f"body{T}.tflite", *flags, runs=5 if "32" in name else 10)
    rows.append({"where": name, "load (s)": load, "per call (s)": infer, "x real time": seconds / infer})
pd.DataFrame(rows).round(2)

,where,load (s),per call (s),x real time
0,"CPU, 1 thread",0.33,3.74,0.86
1,"CPU, 4 threads",0.36,2.30,1.39
2,"CPU, 8 threads",0.38,2.54,1.26
3,"GPU, 16-bit",110.75,0.30,10.64
4,"GPU, 32-bit",373.03,0.81,3.96


The CPU barely keeps up: about 1.4x real time at best, with four threads beating eight, since the Pixel's four
smallest cores slow the others down. The GPU at 16 bits is about seven times faster. This phone needs its GPU for
Kokoro.

Where the CPU's time goes:

In [ ]:
_, _, out = benchmark(f"body{T}.tflite", "--num_threads=4", "--enable_op_profiling=true", runs=2)
summary = out.split("Summary by node type")[-1].splitlines()[1:8]
print("\n".join(line.strip() for line in summary))

[Node type]	  [count]	  [avg ms]	    [avg %]	    [cdf %]	  [mem KB]	[times called]
Convolution (NHWC, F32) IGEMM	       18	  1622.103	    60.769%	    60.769%	     0.000	       60
SIN	       50	   579.774	    21.720%	    82.489%	     0.000	       50
Binary Elementwise (ND)	      178	   254.038	     9.517%	    92.006%	     0.000	      628
Transpose (ND, X32) Transpose	       46	   102.837	     3.853%	    95.858%	     0.000	      134
Deconvolution (NHWC, F32) Subconv2D	        4	    44.236	     1.657%	    97.516%	     0.000	        4
Mean (ND) Mean	       37	    27.148	     1.017%	    98.533%	     0.000	      108


Convolutions, as you'd expect, but a fifth of it is `SIN`. The decoder's residual blocks use the "Snake" activation,
`x + sin(a·x)² / a`, on every sample of audio, and the phone's CPU is slow at sines. The Mac never showed this.

## Loading

The GPU's first load is slow: it compiles programs for the model's shapes, which takes about two minutes at 16 bits
(six at 32). LiteRT can save them to disk, and an app would do this once, at install or first launch:

In [ ]:
cache = ["--use_gpu=true", f"--delegate_serialize_dir={ON_PHONE}/gpu_cache", f"--delegate_serialize_token=body{T}"]
adb("shell", "rm", "-rf", f"{ON_PHONE}/gpu_cache")
adb("shell", "mkdir", "-p", f"{ON_PHONE}/gpu_cache")
for attempt in ["first", "second"]:
    load, infer, _ = benchmark(f"body{T}.tflite", *cache, runs=5)
    print(f"{attempt} load: {load:.1f}s to load, {infer:.2f}s per call")

first load: 175.8s to load, 0.30s per call
second load: 0.6s to load, 0.30s per call


## Summary

- **The decoder converts to LiteRT with five rewrites**, all exact: the converter mishandles padded transposed
  convolutions, splits depthwise ones into a thousand pieces, and the GPU can't take gathers or a constant input.
- **On the Pixel 10a's GPU it runs at about 10x real time**, and about 1.4x on its CPU. Use the GPU.
- **The GPU's first load takes two to three minutes**, and well under a second once its compiled programs are cached.

What isn't measured here: the phone GPU's own output. Accuracy was checked on the Mac's LiteRT CPU, and the 16-bit
GPU path only for speed. Google's tool can't save outputs, so checking it means an app, which is also what measuring
first-sound latency and battery needs.

## Next

- An Android app, starting with notebook 102's player: the front half on the CPU, the decoder on the GPU, Sonic for
  speed. It's also where to check the GPU's output by ear.
- The 214 MB model is 32-bit weights. LiteRT's 16-bit and 8-bit weights would shrink the download as Core ML's did in
  notebook 103.
- The Snake activation's sines are a fifth of the CPU's time. For our own model: a cheaper activation.